In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Null_handling').getOrCreate()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [3]:
from google.colab import files
files.upload()

Saving bookings.csv to bookings.csv


{'bookings.csv': b'booking_id,customer_name,city,service_type,provider,booking_amount,booking_status,payment_mode\r\n1001,Aarav Mehta,Hyderabad,Flight,IndiGo,6500,Confirmed,UPI\r\n1002,Sana Khan,Bangalore,Hotel,Pearl Grand,4500,Confirmed,Card\r\n1003,John Mathew,,Flight,Air India,12000,Confirmed,UPI\r\n1004,Ayesha Begum,Hyderabad,Hotel,,7500,Pending,Cash\r\n1005,Vikram Rao,Mumbai,Flight,Vistara,,Confirmed,Card\r\n1006,Divya Sharma,Delhi,Flight,IndiGo,5900,Cancelled,\r\n1007,Imran Ali,Pune,Hotel,Budget Inn,2200,,UPI\r\n1008,Meera Nair,Kochi,Hotel,Hill View Resort,7500,Confirmed,Card\r\n1009,Rohan Das,Kolkata,Flight,Air India,7400,Pending,UPI\r\n1010,Nisha Reddy,Bangalore,Flight,British Airways,62000,Confirmed,Card\r\n1011,Farhan Ali,,Hotel,Skyline Suites,22000,Confirmed,\r\n1012,Neha Singh,Hyderabad,,Emirates,28000,Confirmed,UPI\r\n1013,Arjun Verma,Chennai,Flight,,15000,Cancelled,Cash\r\n1014,Kavya Nair,Mumbai,Hotel,Sea View Stay,,Pending,Card\r\n1015,Ravi Kumar,Delhi,Flight,SpiceJet,48

In [4]:
#1
df=spark.read.csv("bookings.csv",header=True,inferSchema=True)
df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [5]:
# 2
df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)



In [6]:
# 3
df.count()

15

In [7]:
# 4 - 8
df.filter(df.city.isNull()).show()
df.filter(col("provider").isNull()).show()
df.filter(col("booking_status").isNull()).show()
df.filter(col("booking_amount").isNull()).show()
df.filter(col("payment_mode").isNull()).show()

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|      1003|  John Mathew|NULL|      Flight|     Air India|         12000|     Confirmed|         UPI|
|      1011|   Farhan Ali|NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|      1004| Ayesha Begum|Hyderabad|       Hotel|    NULL|          7500|  

In [8]:
df.select([count(when(col(c).isNull(), c))for c in df.columns]).show()

+---------------------------------------------------------+---------------------------------------------------------------+---------------------------------------------+-------------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------------------+-----------------------------------------------------------------+-------------------------------------------------------------+
|count(CASE WHEN (booking_id IS NULL) THEN booking_id END)|count(CASE WHEN (customer_name IS NULL) THEN customer_name END)|count(CASE WHEN (city IS NULL) THEN city END)|count(CASE WHEN (service_type IS NULL) THEN service_type END)|count(CASE WHEN (provider IS NULL) THEN provider END)|count(CASE WHEN (booking_amount IS NULL) THEN booking_amount END)|count(CASE WHEN (booking_status IS NULL) THEN booking_status END)|count(CASE WHEN (payment_mode IS NULL) THEN payment_mode END)|
+---------------------------------------

In [9]:
drop_any_null_df = df.na.drop()
drop_any_null_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1008|   Meera Nair|    Kochi|       Hotel|Hill View Resort|          7500|     Confirmed|        Card|
|      1009|    Rohan Das|  Kolkata|      Flight|       Air India|          7400|       Pending|         UPI|
|      1010|  Nisha Reddy|Bangalore|      Flight| British Airways|         62000|     Confirmed|        Card|
|      1015|   Ravi Kumar|    Delhi|      Flight|        SpiceJet|          4800|     Confirmed|         UPI|
+---------

In [10]:
df.na.drop(subset=["booking_amount"]).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|         UPI|
|      100

In [11]:
df.na.drop(
    subset=["customer_name","service_type","booking_amount"]
).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|         UPI|
|      100

In [12]:
# 13 - 17
filled_df = df.na.fill(
    {
        "city" : "Unknown",
        "provider" : "Not Available",
        "booking_status" : "Unknown",
        "payment_mode" : "Not Provided",
        "booking_amount" : 0
    }
)
filled_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|Not Provided|
|      100

In [13]:
new_df = df.withColumn(
    "data_quality_status",
    when(
        col("customer_name").isNull() |
        col("service_type").isNull() |
        col("booking_amount").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

new_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|           Complete|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Conf

In [14]:
new_df.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   12|
|         Incomplete|    3|
+-------------------+-----+



In [15]:
# 20
new_df.filter(col("data_quality_status") == "Complete").show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|           Complete|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Canc

In [16]:
# 21
new_df.filter(col("data_quality_status") == "Incomplete").show()

+----------+-------------+---------+------------+-------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|     provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+-------------+--------------+--------------+------------+-------------------+
|      1005|   Vikram Rao|   Mumbai|      Flight|      Vistara|          NULL|     Confirmed|        Card|         Incomplete|
|      1012|   Neha Singh|Hyderabad|        NULL|     Emirates|         28000|     Confirmed|         UPI|         Incomplete|
|      1014|   Kavya Nair|   Mumbai|       Hotel|Sea View Stay|          NULL|       Pending|        Card|         Incomplete|
+----------+-------------+---------+------------+-------------+--------------+--------------+------------+-------------------+



In [17]:
# 22
new_df = new_df.withColumn(
    "tax",
    col("booking_amount") * 0.05
)
new_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|   tax|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete| 325.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete| 225.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete| 600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|           Complete| 375.0|
|      1005|   Vikram Rao|   Mumbai|     

In [18]:
# 23
new_df = new_df.withColumn(
    "final_amount",
    col("booking_amount") + col("tax")
)
new_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|   tax|final_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete| 325.0|      6825.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete| 225.0|      4725.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete| 600.0|     12600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|    

In [19]:
new_df.filter(col("booking_status") == "Confirmed"
).select(sum("final_amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|     154665.0|
+-------------+



In [20]:
# 25
new_df.na.fill({"service_type": "Unknown"
}).groupBy("service_type"
).count().show()

+------------+-----+
|service_type|count|
+------------+-----+
|     Unknown|    1|
|       Hotel|    6|
|      Flight|    8|
+------------+-----+



In [22]:
new_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|     NULL|    2|
|   Mumbai|    2|
|  Kolkata|    1|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [23]:
# 27
new_df.select(avg("booking_amount")).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 14253.846153846154|
+-------------------+



In [24]:
# 28
df.na.drop(subset=["booking_amount"]
).select(avg("booking_amount")
).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 14253.846153846154|
+-------------------+



In [25]:
avg_fill_zero = new_df.select(avg("booking_amount")).collect()[0][0]

avg_drop_null = df.na.drop(
    subset=["booking_amount"]
).select(avg("booking_amount")
).collect()[0][0]

print("Average after filling nulls with 0 :", avg_fill_zero)
print("Average after dropping null rows   :", avg_drop_null)

Average after filling nulls with 0 : 14253.846153846154
Average after dropping null rows   : 14253.846153846154


In [26]:
new_df.write.mode("overwrite").parquet("clean_bookings.parquet")

In [27]:
spark.read.parquet("clean_bookings.parquet").show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|   tax|final_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete| 325.0|      6825.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete| 225.0|      4725.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete| 600.0|     12600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|    

In [29]:
from google.colab import files
files.upload ()

Saving customers.json to customers.json


{'customers.json': b'[ \r\n{ \r\n"customer_id": 1, \r\n"name": "Aarav Mehta", \r\n"city": "Hyderabad", \r\n"membership": "Gold", \r\n"contact": { \r\n"phone": "9876500011", \r\n"email": "aarav@mail.com" \r\n}, \r\n"preferences": {\r\n"preferred_service": "Flight", \r\n"budget_range": "Medium" \r\n}\r\n}, \r\n{ \r\n"customer_id": 2, \r\n"name": "Sana Khan", \r\n"city": "Bangalore", \r\n"membership": "Silver", \r\n"contact": { \r\n"phone": null,\r\n"email": "sana@mail.com" \r\n}, \r\n"preferences": {\r\n"preferred_service": "Hotel", \r\n"budget_range": null \r\n}\r\n}, \r\n{ \r\n"customer_id": 3, \r\n    "name": "John Mathew", \r\n    "city": null, \r\n    "membership": "Gold", \r\n    "contact": { \r\n      "phone": "9876500013", \r\n      "email": null \r\n    }, \r\n    "preferences": {\r\n      "preferred_service": "Flight", \r\n      "budget_range": "High" \r\n    }\r\n  }, \r\n  { \r\n    "customer_id": 4, \r\n    "name": "Ayesha Begum", \r\n    "city": "Hyderabad", \r\n    "member

In [30]:
cdf = spark.read.option("multiline","true").json("customers.json")
cdf.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [31]:
cdf.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)



In [33]:
flat_customers_df = cdf.select(
    "customer_id",
    "name",
    "city",
    "membership",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
)

flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [34]:
flat_customers_df.select("name","city","phone", "email").show()

+------------+---------+----------+---------------+
|        name|     city|     phone|          email|
+------------+---------+----------+---------------+
| Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|   Sana Khan|Bangalore|      NULL|  sana@mail.com|
| John Mathew|     NULL|9876500013|           NULL|
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|  Vikram Rao|   Mumbai|      NULL|           NULL|
+------------+---------+----------+---------------+



In [35]:
# 7
flat_customers_df.filter(col("city").isNull()).show()

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|       name|city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|          3|John Mathew|NULL|      Gold|9876500013| NULL|           Flight|        High|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+



In [36]:
flat_customers_df.filter(col("phone").isNull()).show()

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|      name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|          2| Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
|          5|Vikram Rao|   Mumbai|  Platinum| NULL|         NULL|           Flight|        High|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+



In [37]:
flat_customers_df.filter(col("email").isNull()).show()

+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|customer_id|       name|  city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|          3|John Mathew|  NULL|      Gold|9876500013| NULL|           Flight|        High|
|          5| Vikram Rao|Mumbai|  Platinum|      NULL| NULL|           Flight|        High|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+



In [38]:
flat_customers_df.filter( col("membership").isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [39]:
# 11
flat_customers_df.filter(col("preferred_service").isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [40]:
flat_customers_df.filter(col("budget_range").isNull()).show()

+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|     name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|          2|Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+



In [41]:

flat_customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in flat_customers_df.columns
]).show()


+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [44]:
# 14 - 19
new_cdf = flat_customers_df.na.fill({
    "city": "Unknown",
    "membership": "Standard",
    "phone": "Not Provided",
    "email": "Not Provided",
    "preferred_service": "Not Selected",
    "budget_range": "Unknown"
})
new_cdf.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [45]:
new_cdf = new_cdf.withColumn(
    "customer_quality_status",
    when(
        col("city").isNull() |
        col("phone").isNull() |
        col("email").isNull() |
        col("membership").isNull() |
        col("preferred_service").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

new_cdf.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|               Complete|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|               Complete|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|               Complete|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|

In [46]:
new_cdf.groupBy("customer_quality_status").count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    5|
+-----------------------+-----+



In [47]:
# 22
new_cdf.filter(col("customer_quality_status") == "Complete").show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|               Complete|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|               Complete|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|               Complete|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|

In [48]:
# 23
new_cdf.filter(col("customer_quality_status") == "Incomplete").show()

+-----------+----+----+----------+-----+-----+-----------------+------------+-----------------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|customer_quality_status|
+-----------+----+----+----------+-----+-----+-----------------+------------+-----------------------+
+-----------+----+----+----------+-----+-----+-----------------+------------+-----------------------+



In [49]:
new_cdf.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [50]:
new_cdf.groupBy("preferred_service").count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [51]:
flat_customers_df.write.mode( "overwrite").parquet("customers_flat.parquet")

In [52]:
spark.read.parquet("customers_flat.parquet").show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [53]:
new_cdf.write.mode("overwrite"
).option("header","true").csv("clean_customers.csv")

In [54]:
spark.read.csv("clean_customers.csv").show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+--------------------+
|        _c0|         _c1|      _c2|       _c3|         _c4|            _c5|              _c6|         _c7|                 _c8|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+--------------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_...|
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|            Complete|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|            Complete|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|            Complete|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|   

In [55]:
print("Original Count :", flat_customers_df.count())
print("Clean Count    :", new_cdf.count())

Original Count : 5
Clean Count    : 5


In [56]:

flat_customers_df.filter(col("phone").isNull() |col("email").isNull()
).show()


+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|       name|     city|membership|     phone|        email|preferred_service|budget_range|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|          2|  Sana Khan|Bangalore|    Silver|      NULL|sana@mail.com|            Hotel|        NULL|
|          3|John Mathew|     NULL|      Gold|9876500013|         NULL|           Flight|        High|
|          5| Vikram Rao|   Mumbai|  Platinum|      NULL|         NULL|           Flight|        High|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+



In [57]:
flat_customers_df.filter(col("preferred_service").isNull() |col("budget_range").isNull()
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [58]:
flat_customers_df.filter(col("preferred_service").isNull() ).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+

